# RigTech · runner_colab

Este notebook **não contém lógica**. Ele apenas prepara o ambiente, traz os dados do Drive para o disco local e chama os scripts em `src/`.

Se você se pegar escrevendo lógica dentro de uma célula, leve para `src/` e faça commit.

## Pré-requisito no Drive

Antes de rodar, garanta a estrutura em `Drive/rigtech-weed-cycle/`:

```
Drive/rigtech-weed-cycle/
├── versions/v1/
│   ├── dataset.tar.gz     (437 MB, do repo local: work/versions/v1/)
│   ├── manifest.json
│   └── changelog.json
└── golden/
    ├── images/            (215 tiles Flaviano01, jpg)
    ├── labels/            (215 txts)
    └── background_tiles.json
```

Runtime → Change runtime type → **GPU A100** (T4 se A100 indisponível — dobra o tempo).

## 1. Clonar repo e instalar dependências

In [ ]:
!git clone https://github.com/Rigtech-Solutions/rigtech-weed-cycle.git
%cd rigtech-weed-cycle

In [ ]:
!pip install -q ultralytics==8.4.115 rasterio shapely pyproj pillow pyyaml

## 2. Montar o Drive e trazer o dataset para disco local

**Nunca treine lendo o Drive montado** — a latência por arquivo mata o dataloader. Sempre: copiar tarball -> extrair em `/content/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p work/versions/v1 work/golden work/runs
!cp /content/drive/MyDrive/rigtech-weed-cycle/versions/v1/dataset.tar.gz  work/versions/v1/
!cp /content/drive/MyDrive/rigtech-weed-cycle/versions/v1/manifest.json   work/versions/v1/
!cp /content/drive/MyDrive/rigtech-weed-cycle/versions/v1/changelog.json  work/versions/v1/
!cp -r /content/drive/MyDrive/rigtech-weed-cycle/golden/. work/golden/
!ls -la work/versions/v1/ && echo '---' && ls work/golden/images/ | wc -l

## 3. Treino baseline

Overrides via CLI (config.yaml fica intocado). A100: `batch=16 imgsz=1024 amp=true`. `--device 0` = primeira GPU CUDA.

In [ ]:
!python -m src.train_eval --version v1 --tag baseline_colab \
    --device 0 --batch 16 --imgsz 1024 --amp --epochs 100

## 4. Ver resultado + histórico

In [ ]:
!cat work/runs/history.json

In [ ]:
import glob, os
run_dirs = sorted(glob.glob('work/runs/v1_baseline_colab_*'))
run_dir = run_dirs[-1] if run_dirs else None
print(run_dir)
!ls -la {run_dir}/train/ 2>/dev/null || echo 'sem run_dir'

## 5. Devolver ao Drive ANTES da sessão morrer

A etapa mais frequentemente esquecida. Sem isso, tudo se perde ao encerrar o Colab.

In [ ]:
!mkdir -p /content/drive/MyDrive/rigtech-weed-cycle/runs
!rsync -av work/runs/ /content/drive/MyDrive/rigtech-weed-cycle/runs/
!cp work/runs/history.json /content/drive/MyDrive/rigtech-weed-cycle/